# Visualize Learning Curves

In [ ]:
# Import required libraries
import os
import datetime
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
LINE_WIDTH = 2.5
FONT_SIZE = 20
AXIS_LABEL_FONT_SIZE = 22
LEGEND_FONT_SIZE = 18
FIG_SIZE = (8.0, 4.8)
SOLID_ALPHA = 0.9
SHADED_ALPHA = 0.1
import numpy as np
COLOR_SET = plt.cm.viridis(np.linspace(0, 1, 9))  # Max 9 distinct colors. color_set[0]: vivid <-> color_set[-1]: faded
LINESTYLE_SET = ['-', '--', '-.', (0, (3, 1, 1, 1)), ':', (0, (1, 3)), (0, (5, 1)), (0, (5, 1, 1, 1, 1, 1))]  # Max 8 distinct line styles

plt.rcParams.update(
    {
        'font.size': FONT_SIZE,
        'axes.labelsize': AXIS_LABEL_FONT_SIZE,
        'legend.fontsize': LEGEND_FONT_SIZE,
        'text.usetex': True,
        'axes.linewidth': LINE_WIDTH,
        'xtick.major.width': LINE_WIDTH,
        'ytick.major.width': LINE_WIDTH,
        'xtick.major.size': 2*LINE_WIDTH,
        'ytick.major.size': 2*LINE_WIDTH,
        "lines.linewidth": LINE_WIDTH,
    }
)

In [ ]:
# Save options
SAVE_FIGS = True
SAVE_DIR = os.getcwd()

DATA_DIRS = {
    "BCHG": "../data/local/experiment/DiscreteToy4_R1-v0_BCHGDiscrete_Subopt_reset_q/aggregated",
    "BiAC_on_policy": "../data/local/experiment/DiscreteToy4_R1-v0_BiAC_Subopt_onpolicy_reset_q/aggregated",
    "BiAC_off_policy": "../data/local/experiment/DiscreteToy4_R1-v0_BiAC_Subopt_offpolicy_reset_q/aggregated",
    "NaivePGD_on_policy": "../data/local/experiment/DiscreteToy4_R1-v0_BaselineDiscrete_Subopt_onpolicy_reset_q/aggregated",
    "NaivePGD_off_policy": "../data/local/experiment/DiscreteToy4_R1-v0_BaselineDiscrete_Subopt_offpolicy_reset_q/aggregated",
}  # List of directories containing multiple experimental results with different stop_q_iteration values (reset-q setting)
METRICS = [
    "Evaluation_opt/AverageDiscountedTargetReturn",
    "Follower/OptimalityGap",
]  # Metrics to visualize
SUBOPT_PARAM = "stop_q_iteration"  # Suboptimality parameter name
NAME_TAG = SUBOPT_PARAM  # Tag included in saved file names

# Plot options
ALGO_LABELS = {
    "BCHG": r"\textsc{BC-HG}",
    "BiAC_on_policy": r"\textsc{Bi-AC (On-policy)}",
    "BiAC_off_policy": r"\textsc{Bi-AC}",
    "NaivePGD_on_policy": r"\textsc{Naive-PGD}",
    "NaivePGD_off_policy": r"\textsc{Naive-PGD (Off-policy)}",
}  # Mapping from DATA_DIRS.keys() to labels used in the plot legend (if None, DATA_DIRS.keys() are used as-is)
X_AXIS = 'StepItr'  # Quantity used for the x-axis
MAX_X_VALUE = 500
X_LABEL = rf"Outer iterations: $N$"  # x-axis label
Y_LABELS = {
    "Evaluation_opt/AverageDiscountedTargetReturn": rf"Upper-level objective",
    "Follower/OptimalityGap": rf"Lower-level KL optimality gap",
}   # y-axis label (None means automatic generation)
YSCALE = {
    "Evaluation_opt/AverageDiscountedTargetReturn": ["linear"],
    "Follower/OptimalityGap": ["linear", "log"],
}
ROLLING_WINDOW = 10  # Rolling smoothing window width. If None, no smoothing is applied.
LABELS = None
LINESTYLES = None
COLORS = None

In [ ]:
def find_dirs_end_with(dir, keyword):
    found_dirs = []
    for root, dirs, files in os.walk(dir):
        for d in dirs:
            if d.endswith(keyword):
                found_dirs.append(os.path.join(root, d))
    found_dirs.sort()
    return found_dirs

def find_files_end_with(dir, keyword):
    found_files = []
    for root, dirs, files in os.walk(dir):
        for f in files:
            if f.endswith(keyword):
                found_files.append(os.path.join(root, f))
    found_files.sort()
    return found_files

## Load Data

In [ ]:
# --- Load data --- #
import re

def extract_subopt_param_label(dir_path: str) -> str:
    basename = os.path.basename(dir_path.rstrip("/"))
    match = re.search(rf"{SUBOPT_PARAM}_(.+?)_(?:average|std)$", basename)  # Support "*SUBOPT_PARAM_<VALUE(int/float)>_average/std"
    if match is not None:
        return match.group(1)
    return basename


def subopt_param_sort_key(label: str):
    try:
        return float(label)
    except ValueError:
        return label


ave_data_for_plot = {DATA_DIR_KEY: [] for DATA_DIR_KEY in DATA_DIRS}
std_data_for_plot = {DATA_DIR_KEY: [] for DATA_DIR_KEY in DATA_DIRS}
x_data_for_plot = {DATA_DIR_KEY: [] for DATA_DIR_KEY in DATA_DIRS}
label_list = {DATA_DIR_KEY: [] for DATA_DIR_KEY in DATA_DIRS}
color_id_list = {DATA_DIR_KEY: [] for DATA_DIR_KEY in DATA_DIRS}
linestyle_id_list = {DATA_DIR_KEY: [] for DATA_DIR_KEY in DATA_DIRS}

for DATA_DIR_KEY, DATA_DIR_PATH in DATA_DIRS.items():
    ave_sep_data_list = []
    std_sep_data_list = []

    print('Source log directory:', DATA_DIR_PATH)
    ave_dirs = find_dirs_end_with(DATA_DIR_PATH, 'average')  # Directories of all hyperparameter settings under DATA_DIR
    ave_dirs = [ave_dir for ave_dir in ave_dirs if f"{SUBOPT_PARAM}_" in os.path.basename(ave_dir)]
    ave_dirs.sort(key=lambda ave_dir: subopt_param_sort_key(extract_subopt_param_label(ave_dir)), reverse=True)

    if not len(ave_dirs) > 0:
        raise FileNotFoundError(f"No directory with '{SUBOPT_PARAM}_*_average' found in {DATA_DIR_PATH}")

    for ave_dir in ave_dirs:
        subopt_param_label = extract_subopt_param_label(ave_dir)
        std_dir = ave_dir.replace('_average', '_std')

        ave_paths = find_files_end_with(ave_dir, 'progress_average.csv')
        std_paths = find_files_end_with(std_dir, 'progress_std.csv')

        if not (len(ave_paths) > 0 and len(std_paths) > 0):
            raise FileNotFoundError(f"No progress_average.csv or progress_std.csv found in {ave_dir}")

        # Load
        ave_data = [pd.read_csv(p) for p in ave_paths]  # [eval_DataFrame, follower_DataFrame, leader_DataFrame]
        std_data = [pd.read_csv(p) for p in std_paths]  # [eval_DataFrame, follower_DataFrame, leader_DataFrame]
        ave_sep_data_list.append(ave_data)
        std_sep_data_list.append(std_data)

        ave_data_for_plot[DATA_DIR_KEY].append({})
        std_data_for_plot[DATA_DIR_KEY].append({})
        x_data_for_plot[DATA_DIR_KEY].append({})
        label_list[DATA_DIR_KEY].append(subopt_param_label)
        color_id_list[DATA_DIR_KEY].append(len(label_list[DATA_DIR_KEY]) - 1)
        linestyle_id_list[DATA_DIR_KEY].append(len(label_list[DATA_DIR_KEY]) - 1)

    # --- Extract data to plot --- #

    for curve_idx, (ave_data, std_data, ave_dir) in enumerate(zip(ave_sep_data_list, std_sep_data_list, ave_dirs)):
        ave_for_metrics = {}
        std_for_metrics = {}
        x_for_metrics = {}
        extracted = []
        for metric in METRICS:
            for ave, std in zip(ave_data, std_data):
                if metric in ave.columns:
                    if metric in extracted:
                        raise ValueError(f"Multiple data found for {metric} in {ave_dir}")
                    ave_for_metrics[metric] = ave[metric]
                    std_for_metrics[metric] = std[metric]
                    x_for_metrics[metric] = ave[X_AXIS]
                    extracted.append(metric)

        ave_data_for_plot[DATA_DIR_KEY][curve_idx] = ave_for_metrics
        std_data_for_plot[DATA_DIR_KEY][curve_idx] = std_for_metrics
        x_data_for_plot[DATA_DIR_KEY][curve_idx] = x_for_metrics

"""
Hint:
- ave_data_for_plot: {DATA_DIR: [ave_for_ratio1, ave_for_ratio2, ...], ...}
    - ave_for_ratio*: {metric1: ave_data1 (pandas.Series), metric2: ave_data2 (pandas.Series), ...}
- std_data_for_plot: {DATA_DIR: [std_for_ratio1, std_for_ratio2, ...], ...}
    - std_for_ratio*: {metric1: std_data1 (pandas.Series), metric2: std_data2 (pandas.Series), ...}
- label_list: {DATA_DIR: [param_label_for_ratio1, param_label_for_ratio2, ...], ...}
    - param_label_for_ratio*: str (value corresponding to {SUBOPT_PARAM})
"""

In [ ]:
def plot_average_with_std(
        ave_data: list, std_data: list, x_data: list, metric: str,
        labels: list = None, colors: list = None, linestyles: list = None,
        solid_alpha: float = SOLID_ALPHA, shaded_alpha: float = SHADED_ALPHA,
        xlim: tuple = None, ylim: tuple = None, yticks: list = None,
        x_label: str = 'Steps', y_label: str = None,
        title: str = None, show_legend: bool = True, legend_position: dict = None,
        save_figure: bool = False, tag: str = '', save_format: str = 'pdf',
        distinct_fill_between_edge: bool = True,
        y_scale: str = 'linear',
        rolling_window: int = None,
    ):
    fig, ax = plt.subplots(figsize=FIG_SIZE)
    labels = [i for i in range(len(ave_data))] if labels is None else labels
    colors = [COLOR_SET[i % len(COLOR_SET)] for i in range(len(ave_data))] if colors is None else colors
    linestyles = [LINESTYLE_SET[i % len(LINESTYLE_SET)] for i in range(len(ave_data))] if linestyles is None else linestyles
    if xlim is None:
        xlim = (0, MAX_X_VALUE)

    def _apply_rolling(series):
        if rolling_window is None or rolling_window <= 1:
            return series
        return series.rolling(window=rolling_window, min_periods=1, center=True).mean()

    for ave, std, x, label, color, linestyle in zip(ave_data, std_data, x_data, labels, colors, linestyles):
        ave_y = _apply_rolling(ave[metric])
        std_y = _apply_rolling(std[metric])
        ax.plot(x[metric], ave_y, label=label, color=color, alpha=solid_alpha, linestyle=linestyle)
        if distinct_fill_between_edge:
            ax.fill_between(x[metric], ave_y - std_y, ave_y + std_y, color=color, alpha=shaded_alpha,
                            edgecolor=color, linewidth=LINE_WIDTH, linestyle=linestyle)
        else:
            ax.fill_between(x[metric], ave_y - std_y, ave_y + std_y, color=color, alpha=shaded_alpha)

    ax.set_xlabel(x_label)
    ax.set_ylabel(metric if y_label is None else y_label)
    if y_scale in ('log', 'symlog'):
        ax.set_yscale(y_scale)
    if title is not None:
        ax.set_title(title)
    if show_legend:
        if legend_position['loc'] == 'outer':
            ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
        elif legend_position is not None:
            ax.legend(**legend_position)
    ax.set_xlim(*xlim)
    if yticks is not None:
        ax.set_yticks(yticks)
    if ylim is not None:
        ax.set_ylim(*ylim)
    plt.show()

    if save_figure:
        _log_dir = os.getcwd()
        _save_dir = SAVE_DIR if SAVE_DIR is not None else os.path.join(_log_dir, 'figures')
        os.makedirs(_save_dir, exist_ok=True)

        metric_label = metric.replace('/', '_')
        suffix = f'_{tag}' if tag != '' else ''
        file_ext = save_format.lower().lstrip('.')
        if file_ext not in ('pdf', 'png'):
            raise ValueError("save_format must be 'pdf' or 'png'")

        file_name = f'{metric_label}_AveStd{suffix}.{file_ext}'
        fig.savefig(os.path.join(_save_dir, file_name), bbox_inches='tight')
        print('Saved at', os.path.join(_save_dir, file_name))

    info = {'xlim': ax.get_xlim(), 'ylim': ax.get_ylim(), 'yticks': ax.get_yticks().tolist()}
    plt.close(fig)
    return info

def color_linestyle_label_matching(label_all, color_id_all, linestyle_id_all):
    # color and linestyle matching
    colors = []
    linestyles = []
    labels = []
    for label, ci, ls in zip(label_all, color_id_all, linestyle_id_all):
        color = COLOR_SET[ci] if 0 <= ci < len(COLOR_SET) else COLOR_SET[0]
        if COLORS is not None:
            for ks, c in COLORS.items():
                if all([k in label for k in ks]):
                    color = c
                    break
        colors.append(color)

        linestyle = LINESTYLE_SET[ls % len(LINESTYLE_SET)] if 0 <= ls < len(LINESTYLE_SET) else LINESTYLE_SET[0]
        if LINESTYLES is not None:
            for ks, s in LINESTYLES.items():
                if all([k in label for k in ks]):
                    linestyle = s
                    break
        linestyles.append(linestyle)

        if LABELS is not None:
            for ks, l in LABELS.items():
                if all([k in label for k in ks]):
                    label = l
                    break
        elif label.endswith('_average'):
            label = label[:-len('_average')]
        labels.append(label)
    return colors, linestyles, labels

## Plot data included in INCLUDE within a single figure

In [ ]:
# --- Plot & notebook display --- #
LEGEND_POSITION = {
    'loc': 'center right',  # outer: place the legend at a fixed position outside the figure; upper right: top right; upper left: top left; lower right: bottom right; lower left: bottom left
    'bbox_to_anchor': (0.975, 0.5),
    'ncol': 1
}
ylims = {
    ("Evaluation_opt/AverageDiscountedTargetReturn", "linear"): (-7, 35),
    ("Follower/OptimalityGap", "linear"): (-1.0, 7.5),
    ("Follower/OptimalityGap", "log"): (7e-9, 15.0),
}

for metric in METRICS:
    for yscale in YSCALE.get(metric, ['linear']):
        print(f'\n* --- Mertric: {metric} ({yscale}) --- *')
        info = {'xlim': None, 'yticks': None}

        for DATA_DIR_KEY, DATA_DIR_PATH in DATA_DIRS.items():
            if not len(label_list[DATA_DIR_KEY]) > 0:
                continue
            title = ALGO_LABELS.get(DATA_DIR_KEY, DATA_DIR_KEY)
            print(f'\n=== DATA_DIR: {DATA_DIR_KEY} ===')
            colors, linestyles, labels = color_linestyle_label_matching(label_list[DATA_DIR_KEY], 
                                                                        color_id_list[DATA_DIR_KEY], 
                                                                        linestyle_id_list[DATA_DIR_KEY])
            tag = DATA_DIR_KEY if NAME_TAG == '' else f'{DATA_DIR_KEY}_{NAME_TAG}'
            tag = tag + f'_logscale' if yscale in ('log', 'symlog') else tag
            info = plot_average_with_std(
                ave_data=ave_data_for_plot[DATA_DIR_KEY],
                std_data=std_data_for_plot[DATA_DIR_KEY],
                x_data=x_data_for_plot[DATA_DIR_KEY],
                labels=labels,
                metric=metric,
                colors=colors,
                linestyles=linestyles,
                xlim=info['xlim'],
                ylim=ylims.get((metric, yscale), None),
                yticks=info['yticks'],
                x_label=X_LABEL,
                y_label=Y_LABELS[metric],
                title=title,
                show_legend=True,
                legend_position=LEGEND_POSITION,
                save_figure=SAVE_FIGS,
                tag=tag,
                y_scale=yscale,
                rolling_window=ROLLING_WINDOW,
            )